# Train AlexNet like Model

## 1. Setup

### Imports, paths and download data

In [2]:
import sys
print(sys.executable)

/home/rafael/Documents/alexnet_rafael/.venv/bin/python


In [3]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install tqdm optuna kagglehub

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from tqdm.auto import tqdm

import kagglehub
import optuna

# =========================================================
# GPU SETTINGS
# =========================================================

torch.backends.cudnn.benchmark = True

torch.set_float32_matmul_precision("high")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# =========================================================
# SAVE DIRECTORY
# =========================================================

SAVE_DIR = "./saved_models"

os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Models will be saved to: {SAVE_DIR}")

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Models will be saved to: ./saved_models


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())

print("GPU count:", torch.cuda.device_count())

print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
dataset_path = kagglehub.dataset_download(
    "akash2sharma/tiny-imagenet"
)

print("Dataset downloaded to:")
print(dataset_path)

train_path = os.path.join(
    dataset_path,
    "tiny-imagenet-200",
    "train"
)

val_path = os.path.join(
    dataset_path,
    "tiny-imagenet-200",
    "val"
)

print("\nTrain path exists:", os.path.exists(train_path))
print("Val path exists:", os.path.exists(val_path))

Dataset downloaded to:
/home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1

Train path exists: True
Val path exists: True


### Data Preprocessing

In [7]:
# =========================================================
# SETTINGS
# =========================================================

BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10

# =========================================================
# TRANSFORMS
# =========================================================

transform_train = transforms.Compose([

    transforms.RandomResizedCrop(
        64,
        scale=(0.75, 1.0)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        0.2,
        0.2,
        0.2
    ),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_val = transforms.Compose([

    transforms.Resize(64),

    transforms.CenterCrop(64),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================================================
# DATASETS
# =========================================================

full_train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=transform_train
)

train_size = int(
    0.9 * len(full_train_dataset)
)

val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

# =========================================================
# DATALOADERS
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("DataLoaders ready.")

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {len(full_train_dataset.classes)}")

DataLoaders ready.
Train samples: 90000
Validation samples: 10000
Classes: 200


### Generic Training Function

In [8]:
def train_model(
    model,
    train_loader,
    val_loader,
    epochs,
    optimizer,
    scheduler,
    criterion,
    save_name
):

    scaler = torch.amp.GradScaler("cuda")

    best_accuracy = 0

    model.to(device)

    for epoch in range(epochs):

        epoch_start = time.time()

        model.train()

        running_loss = 0

        correct = 0
        total = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs}"
        )

        for images, labels in progress_bar:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)

            correct += (
                predictions == labels
            ).sum().item()

            current_acc = 100 * correct / total

            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{current_acc:.2f}%"
            })

        if scheduler is not None:
            scheduler.step()

        train_accuracy = 100 * correct / total

        # =====================================================
        # VALIDATION
        # =====================================================

        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)

                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                val_total += labels.size(0)

                val_correct += (
                    predictions == labels
                ).sum().item()

        val_accuracy = 100 * val_correct / val_total

        print(f"\nTrain Accuracy: {train_accuracy:.2f}%")

        print(f"Validation Accuracy: {val_accuracy:.2f}%")

        if val_accuracy > best_accuracy:

            best_accuracy = val_accuracy

            torch.save(

                model.state_dict(),

                os.path.join(
                    SAVE_DIR,
                    save_name
                )
            )

            print("Best model saved.")

        print(
            f"Epoch Time: "
            f"{time.time() - epoch_start:.1f}s"
        )

    return best_accuracy

## 2. Baseline: FastTinyCNN

In [11]:
class FastTinyCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.5),

            nn.Linear(128, 200)
        )

    def forward(self, x):

        return self.classifier(
            self.features(x)
        )

In [11]:
model = FastTinyCNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = None

train_model(
    model,
    train_loader,
    val_loader,
    epochs=10,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    save_name="fast_tiny_cnn.pth"
)

/tmp/ipykernel_9806/2415864544.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/10:   0%|          | 0/1407 [00:00<?, ?it/s]/tmp/ipykernel_9806/2415864544.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/10: 100%|██████████| 1407/1407 [00:34<00:00, 40.26it/s, loss=4.5125, acc=3.90%]



Train Accuracy: 3.90%
Validation Accuracy: 6.40%
Best model saved.
Epoch Time: 39.1s


Epoch 2/10: 100%|██████████| 1407/1407 [00:26<00:00, 52.74it/s, loss=4.5032, acc=7.45%]



Train Accuracy: 7.45%
Validation Accuracy: 10.57%
Best model saved.
Epoch Time: 29.9s


Epoch 3/10: 100%|██████████| 1407/1407 [00:34<00:00, 41.33it/s, loss=4.4320, acc=9.23%]


...

Epoch 9/10: 100%|██████████| 1407/1407 [00:16<00:00, 87.15it/s, loss=4.6763, acc=13.67%]



Train Accuracy: 13.67%
Validation Accuracy: 17.55%
Best model saved.
Epoch Time: 17.7s


Epoch 10/10: 100%|██████████| 1407/1407 [00:15<00:00, 93.51it/s, loss=4.7981, acc=14.24%] 



Train Accuracy: 14.24%
Validation Accuracy: 16.30%
Epoch Time: 16.6s


17.55

## 3. DynamicTinyCNN

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [13]:
class DynamicTinyCNN(nn.Module):

    def __init__(
        self,
        num_classes,
        channels,
        kernel_sizes,
        dropout
    ):

        super().__init__()

        layers = []

        in_channels = 3

        for out_channels, kernel_size in zip(
            channels,
            kernel_sizes
        ):

            layers.extend([

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size,
                    padding=kernel_size // 2
                ),

                nn.BatchNorm2d(out_channels),

                nn.ReLU(inplace=True),

                nn.MaxPool2d(2)
            ])

            in_channels = out_channels

        layers.append(
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(dropout),

            nn.Linear(in_channels, num_classes)
        )

    def forward(self, x):

        return self.classifier(
            self.features(x)
        )

In [14]:
def objective(trial):

    num_layers = trial.suggest_int(
        "num_layers",
        2,
        5
    )

    channels = [

        trial.suggest_categorical(
            f"channels_{i}",
            [32, 64, 128, 256]
        )

        for i in range(num_layers)
    ]

    dropout = trial.suggest_float(
        "dropout",
        0.2,
        0.7
    )

    lr = trial.suggest_float(
        "lr",
        1e-4,
        1e-2,
        log=True
    )

    model = DynamicTinyCNN(
        200,
        channels,
        [3] * num_layers,
        dropout
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr
    )

    criterion = nn.CrossEntropyLoss()

    EPOCHS = 3

    # =====================================================
    # TRAINING LOOP
    # =====================================================

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0

        correct = 0
        total = 0

        train_bar = tqdm(
            train_loader,
            desc=(
                f"Trial {trial.number} | "
                f"Epoch {epoch+1}/{EPOCHS}"
            ),
            leave=False
        )

        for images, labels in train_bar:

            images = images.to(device)

            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)

            correct += (
                predictions == labels
            ).sum().item()

            train_acc = 100 * correct / total

            train_bar.set_postfix({

                "loss": f"{loss.item():.4f}",

                "acc": f"{train_acc:.2f}%"
            })

        # =====================================================
        # VALIDATION
        # =====================================================

        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)

                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                val_total += labels.size(0)

                val_correct += (
                    predictions == labels
                ).sum().item()

        val_acc = 100 * val_correct / val_total

        avg_loss = running_loss / len(train_loader)

        # =====================================================
        # EPOCH PRINT
        # =====================================================

        print("\n" + "-" * 60)

        print(
            f"Trial {trial.number} | "
            f"Epoch {epoch+1}/{EPOCHS}"
        )

        print(f"Train Loss : {avg_loss:.4f}")

        print(f"Train Acc  : {train_acc:.2f}%")

        print(f"Val Acc    : {val_acc:.2f}%")

        print("-" * 60)

        # =====================================================
        # REPORT TO OPTUNA
        # =====================================================

        trial.report(
            val_acc,
            epoch
        )

        if trial.should_prune():

            raise optuna.exceptions.TrialPruned()

    del model

    torch.cuda.empty_cache()

    return val_acc

In [22]:
from tqdm.auto import tqdm
import optuna

# =========================================================
# CREATE STUDY
# =========================================================

study = optuna.create_study(
    direction="maximize"
)

# =========================================================
# PROGRESS BAR
# =========================================================

N_TRIALS = 15

pbar = tqdm(
    total=N_TRIALS,
    desc="Optuna Progress"
)

# =========================================================
# CALLBACK FUNCTION
# =========================================================

def optuna_callback(study, trial):

    pbar.update(1)

    pbar.set_postfix({

        "best_acc": f"{study.best_value:.4f}",

        "last_trial": trial.number
    })

    print("\n" + "=" * 60)

    print(f"Trial {trial.number} finished")

    print(f"Value: {trial.value:.4f}")

    print("Parameters:")

    for key, value in trial.params.items():

        print(f"  {key}: {value}")

    print("=" * 60)

# =========================================================
# RUN OPTUNA
# =========================================================

study.optimize(
    objective,
    n_trials=N_TRIALS,
    callbacks=[optuna_callback]
)

pbar.close()

# =========================================================
# BEST RESULTS
# =========================================================

print("\n" + "#" * 60)

print("BEST PARAMETERS")

print("#" * 60)

print(f"Best Accuracy: {study.best_value:.4f}")

for key, value in study.best_params.items():

    print(f"{key}: {value}")

[I 2026-05-26 16:31:52,037] A new study created in memory with name: no-name-75dc2d14-7dcf-437a-9489-dcdbfa538e9b
Optuna Progress:   0%|          | 0/15 [01:09<?, ?it/s]



------------------------------------------------------------
Trial 0 | Epoch 1/3
Train Loss : 4.8639
Train Acc  : 3.84%
Val Acc    : 7.45%
------------------------------------------------------------



------------------------------------------------------------
Trial 0 | Epoch 2/3
Train Loss : 4.5198
Train Acc  : 6.83%
Val Acc    : 10.92%
------------------------------------------------------------


[I 2026-05-26 16:36:07,799] Trial 0 finished with value: 11.43 and parameters: {'num_layers': 3, 'channels_0': 64, 'channels_1': 256, 'channels_2': 32, 'dropout': 0.34679381773500384, 'lr': 0.009725258515232775}. Best is trial 0 with value: 11.43.
Optuna Progress:   7%|▋         | 1/15 [04:15<59:40, 255.76s/it, best_acc=11.4300, last_trial=0]


------------------------------------------------------------
Trial 0 | Epoch 3/3
Train Loss : 4.3977
Train Acc  : 8.26%
Val Acc    : 11.43%
------------------------------------------------------------

Trial 0 finished
Value: 11.4300
Parameters:
  num_layers: 3
  channels_0: 64
  channels_1: 256
  channels_2: 32
  dropout: 0.34679381773500384
  lr: 0.009725258515232775


...


------------------------------------------------------------
Trial 13 | Epoch 3/3
Train Loss : 3.8178
Train Acc  : 16.05%
Val Acc    : 20.65%
------------------------------------------------------------

Trial 13 finished
Value: 20.6500
Parameters:
  num_layers: 5
  channels_0: 32
  channels_1: 32
  channels_2: 128
  channels_3: 256
  channels_4: 128
  dropout: 0.4423705645462558
  lr: 0.001753943824441896



------------------------------------------------------------
Trial 14 | Epoch 1/3
Train Loss : 4.5360
Train Acc  : 7.12%
Val Acc    : 12.70%
------------------------------------------------------------



------------------------------------------------------------
Trial 14 | Epoch 2/3
Train Loss : 3.8550
Train Acc  : 15.58%
Val Acc    : 19.43%
------------------------------------------------------------


[I 2026-05-26 17:28:05,977] Trial 14 finished with value: 22.7 and parameters: {'num_layers': 5, 'channels_0': 32, 'channels_1': 32, 'channels_2': 128, 'channels_3': 256, 'channels_4': 128, 'dropout': 0.24189005555363188, 'lr': 0.0019189436183285494}. Best is trial 14 with value: 22.7.
Optuna Progress: 100%|██████████| 15/15 [56:13<00:00, 224.93s/it, best_acc=22.7000, last_trial=14]


------------------------------------------------------------
Trial 14 | Epoch 3/3
Train Loss : 3.5697
Train Acc  : 20.28%
Val Acc    : 22.70%
------------------------------------------------------------

Trial 14 finished
Value: 22.7000
Parameters:
  num_layers: 5
  channels_0: 32
  channels_1: 32
  channels_2: 128
  channels_3: 256
  channels_4: 128
  dropout: 0.24189005555363188
  lr: 0.0019189436183285494

############################################################
BEST PARAMETERS
############################################################
Best Accuracy: 22.7000
num_layers: 5
channels_0: 32
channels_1: 32
channels_2: 128
channels_3: 256
channels_4: 128
dropout: 0.24189005555363188
lr: 0.0019189436183285494


## 4. StrongCNN

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [16]:
class ResidualBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):

        out = torch.relu(
            self.bn1(self.conv1(x))
        )

        out = self.bn2(
            self.conv2(out)
        )

        out += self.shortcut(x)

        return torch.relu(out)

In [17]:
class StrongCNN(nn.Module):

    def __init__(self, num_classes=200):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(
                3,
                64,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(64),

            nn.ReLU()
        )

        self.layer1 = ResidualBlock(64, 64)

        self.layer2 = ResidualBlock(
            64,
            128,
            stride=2
        )

        self.layer3 = ResidualBlock(
            128,
            128
        )

        self.layer4 = ResidualBlock(
            128,
            256,
            stride=2
        )

        self.layer5 = ResidualBlock(
            256,
            256
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.layer1(x)

        x = self.layer2(x)

        x = self.layer3(x)

        x = self.layer4(x)

        x = self.layer5(x)

        x = self.pool(x)

        return self.classifier(x)

In [26]:
model = StrongCNN()

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

train_model(
    model,
    train_loader,
    val_loader,
    epochs=20,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    save_name="strong_cnn.pth"
)

/tmp/ipykernel_9806/2415864544.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/20:   0%|          | 0/1407 [00:00<?, ?it/s]/tmp/ipykernel_9806/2415864544.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/20: 100%|██████████| 1407/1407 [01:02<00:00, 22.40it/s, loss=4.4508, acc=5.73%]



Train Accuracy: 5.73%
Validation Accuracy: 10.50%
Best model saved.
Epoch Time: 67.1s


Epoch 2/20: 100%|██████████| 1407/1407 [01:02<00:00, 22.56it/s, loss=4.3612, acc=12.72%]



Train Accuracy: 12.72%
Validation Accuracy: 15.09%
Best model saved.
Epoch Time: 66.4s


Epoch 3/20: 100%|██████████| 1407/1407 [01:02<00:00, 22.59it/s, loss=4.1730, acc=17.36%]


...

Epoch 19/20: 100%|██████████| 1407/1407 [01:02<00:00, 22.57it/s, loss=2.8527, acc=44.15%]



Train Accuracy: 44.15%
Validation Accuracy: 44.18%
Best model saved.
Epoch Time: 66.4s


Epoch 20/20: 100%|██████████| 1407/1407 [01:02<00:00, 22.48it/s, loss=2.7266, acc=44.37%]



Train Accuracy: 44.37%
Validation Accuracy: 43.69%
Epoch Time: 66.6s


44.18

## 5. ImprovedTinyCNN

In [9]:
class SmallResidualBlock(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(channels),

            nn.GELU(),

            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(channels)
        )

    def forward(self, x):

        return torch.nn.functional.gelu(
            x + self.block(x)
        )

In [10]:
class ImprovedTinyCNN(nn.Module):

    def __init__(self, num_classes=200):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                64,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(64),

            nn.GELU(),

            SmallResidualBlock(64),

            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(128),

            nn.GELU(),

            SmallResidualBlock(128),

            nn.MaxPool2d(2),

            nn.Conv2d(
                128,
                256,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(256),

            nn.GELU(),

            SmallResidualBlock(256),

            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.25),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        return self.classifier(
            self.features(x)
        )

In [29]:
model = ImprovedTinyCNN()

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

train_model(
    model,
    train_loader,
    val_loader,
    epochs=20,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    save_name="improved_tiny_cnn.pth"
)

/tmp/ipykernel_9806/2415864544.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/20:   0%|          | 0/1407 [00:00<?, ?it/s]/tmp/ipykernel_9806/2415864544.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/20: 100%|██████████| 1407/1407 [00:51<00:00, 27.48it/s, loss=4.3677, acc=6.86%]



Train Accuracy: 6.86%
Validation Accuracy: 12.95%
Best model saved.
Epoch Time: 54.6s


Epoch 2/20: 100%|██████████| 1407/1407 [00:51<00:00, 27.57it/s, loss=4.6321, acc=14.84%]



Train Accuracy: 14.84%
Validation Accuracy: 17.50%
Best model saved.
Epoch Time: 54.3s


Epoch 3/20: 100%|██████████| 1407/1407 [00:51<00:00, 27.58it/s, loss=3.8418, acc=19.81%]


...

Epoch 19/20: 100%|██████████| 1407/1407 [00:51<00:00, 27.52it/s, loss=2.8015, acc=45.42%]



Train Accuracy: 45.42%
Validation Accuracy: 44.54%
Best model saved.
Epoch Time: 54.5s


Epoch 20/20: 100%|██████████| 1407/1407 [00:51<00:00, 27.44it/s, loss=2.5261, acc=46.02%]



Train Accuracy: 46.02%
Validation Accuracy: 44.75%
Best model saved.
Epoch Time: 54.6s


44.75

In [11]:
import torch

# Create the model architecture
model = ImprovedTinyCNN()

# Load weights
state_dict = torch.load(
    "./saved_models/improved_tiny_cnn.pth",
    map_location="cpu"
)

model.load_state_dict(state_dict)

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

/tmp/ipykernel_49015/274967316.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(


Accuracy: 0.5071


## 6. AlexNet Finetuning

In [20]:
from torchvision.models import alexnet

# =========================================================
# LOAD PRETRAINED ALEXNET
# =========================================================

model = alexnet(weights="IMAGENET1K_V1")

# =========================================================
# REPLACE FINAL CLASSIFIER
# =========================================================

model.classifier[6] = nn.Linear(
    4096,
    200
)

model = model.to(device)

print(model)

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [21]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(

    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),

    lr=1e-4,

    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=10
)

In [32]:
alexnet_accuracy = train_model(

    model=model,

    train_loader=train_loader,

    val_loader=val_loader,

    epochs=10,

    optimizer=optimizer,

    scheduler=scheduler,

    criterion=criterion,

    save_name="alexnet_finetuned.pth"
)

print(
    f"Best AlexNet Accuracy: "
    f"{alexnet_accuracy:.2f}%"
)

/tmp/ipykernel_9806/2415864544.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/10:   0%|          | 0/1407 [00:00<?, ?it/s]/tmp/ipykernel_9806/2415864544.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/10: 100%|██████████| 1407/1407 [00:46<00:00, 30.58it/s, loss=3.2155, acc=20.78%]



Train Accuracy: 20.78%
Validation Accuracy: 26.35%
Best model saved.
Epoch Time: 48.2s


Epoch 2/10: 100%|██████████| 1407/1407 [00:45<00:00, 30.71it/s, loss=3.6318, acc=29.24%]



Train Accuracy: 29.24%
Validation Accuracy: 29.12%
Best model saved.
Epoch Time: 47.8s


Epoch 3/10: 100%|██████████| 1407/1407 [00:45<00:00, 30.64it/s, loss=3.3588, acc=33.15%]


...


Train Accuracy: 47.98%
Validation Accuracy: 36.22%
Best model saved.
Epoch Time: 48.8s


Epoch 9/10: 100%|██████████| 1407/1407 [00:46<00:00, 30.28it/s, loss=2.0633, acc=50.27%]



Train Accuracy: 50.27%
Validation Accuracy: 36.18%
Epoch Time: 48.3s


Epoch 10/10: 100%|██████████| 1407/1407 [00:45<00:00, 30.68it/s, loss=3.1164, acc=51.09%]



Train Accuracy: 51.09%
Validation Accuracy: 36.33%
Best model saved.
Epoch Time: 48.0s
Best AlexNet Accuracy: 36.33%


In [22]:
from torchvision.models import alexnet

# =========================================================
# REBUILD MODEL
# =========================================================

loaded_model = alexnet(
    weights=None
)

loaded_model.classifier[6] = nn.Linear(
    4096,
    200
)

loaded_model.load_state_dict(

    torch.load(

        os.path.join(
            SAVE_DIR,
            "alexnet_finetuned.pth"
        ),

        map_location=device
    )
)

loaded_model = loaded_model.to(device)

loaded_model.eval()

print("AlexNet loaded successfully.")

/tmp/ipykernel_41159/2718089839.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


AlexNet loaded successfully.


In [23]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = loaded_model(images)

        predictions = outputs.argmax(dim=1)

        total += labels.size(0)

        correct += (
            predictions == labels
        ).sum().item()

accuracy = 100 * correct / total

print(
    f"AlexNet Validation Accuracy: "
    f"{accuracy:.2f}%"
)

AlexNet Validation Accuracy: 53.84%


## 7. AlexNet 3X3

In [ ]:
class AlexNet3x3(nn.Module):

    def __init__(self, num_classes=200):

        super().__init__()

        self.features = nn.Sequential(

            # =====================================================
            # BLOCK 1
            # =====================================================

            nn.Conv2d(
                3,
                64,
                kernel_size=3,
                stride=2,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.MaxPool2d(
                kernel_size=2
            ),

            # =====================================================
            # BLOCK 2
            # =====================================================

            nn.Conv2d(
                64,
                192,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.MaxPool2d(
                kernel_size=2
            ),

            # =====================================================
            # BLOCK 3
            # =====================================================

            nn.Conv2d(
                192,
                384,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            # =====================================================
            # BLOCK 4
            # =====================================================

            nn.Conv2d(
                384,
                256,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            # =====================================================
            # BLOCK 5
            # =====================================================

            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.MaxPool2d(
                kernel_size=2
            ),

            nn.AdaptiveAvgPool2d((6,6))
        )

        self.classifier = nn.Sequential(

            nn.Dropout(0.5),

            nn.Linear(
                256 * 6 * 6,
                4096
            ),

            nn.ReLU(inplace=True),

            nn.Dropout(0.5),

            nn.Linear(
                4096,
                4096
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                4096,
                num_classes
            )
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x

In [25]:
model = AlexNet3x3(
    num_classes=200
).to(device)

print(model)

AlexNet3x3(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (13): AdaptiveAvgPool2d(output_size=(6, 6))
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
   

In [26]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(

    model.parameters(),

    lr=3e-4,

    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=20
)

In [38]:
alexnet3x3_accuracy = train_model(

    model=model,

    train_loader=train_loader,

    val_loader=val_loader,

    epochs=20,

    optimizer=optimizer,

    scheduler=scheduler,

    criterion=criterion,

    save_name="alexnet3x3.pth"
)

print(
    f"AlexNet3x3 Accuracy: "
    f"{alexnet3x3_accuracy:.2f}%"
)

/tmp/ipykernel_9806/2415864544.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/20:   0%|          | 0/1407 [00:00<?, ?it/s]/tmp/ipykernel_9806/2415864544.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/20: 100%|██████████| 1407/1407 [00:46<00:00, 30.46it/s, loss=4.9952, acc=2.23%]



Train Accuracy: 2.23%
Validation Accuracy: 5.08%
Best model saved.
Epoch Time: 48.5s


Epoch 2/20: 100%|██████████| 1407/1407 [00:45<00:00, 30.66it/s, loss=4.0039, acc=7.83%]



Train Accuracy: 7.83%
Validation Accuracy: 9.99%
Best model saved.
Epoch Time: 48.0s


Epoch 3/20: 100%|██████████| 1407/1407 [00:46<00:00, 30.55it/s, loss=4.3329, acc=12.84%]


...


Train Accuracy: 39.15%
Validation Accuracy: 32.85%
Best model saved.
Epoch Time: 48.2s


Epoch 19/20: 100%|██████████| 1407/1407 [00:45<00:00, 30.62it/s, loss=3.4207, acc=40.00%]



Train Accuracy: 40.00%
Validation Accuracy: 32.64%
Epoch Time: 47.8s


Epoch 20/20: 100%|██████████| 1407/1407 [00:45<00:00, 30.61it/s, loss=3.1290, acc=39.98%]



Train Accuracy: 39.98%
Validation Accuracy: 33.26%
Best model saved.
Epoch Time: 48.2s
AlexNet3x3 Accuracy: 33.26%


In [27]:
loaded_model = AlexNet3x3(
    num_classes=200
).to(device)

loaded_model.load_state_dict(

    torch.load(

        os.path.join(
            SAVE_DIR,
            "alexnet3x3.pth"
        ),

        map_location=device
    )
)

loaded_model.eval()

print("AlexNet3x3 loaded.")

AlexNet3x3 loaded.


/tmp/ipykernel_41159/3085798289.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


## 8. Ranking Models

In [29]:
def evaluate_model(model, loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return 100.0 * correct / total

In [30]:
models = {}
results = {}

In [31]:
models["FastTinyCNN"] = FastTinyCNN()

models["FastTinyCNN"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "fast_tiny_cnn.pth"), map_location=device)
)

/tmp/ipykernel_41159/1647010323.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "fast_tiny_cnn.pth"), map_location=device)


<All keys matched successfully>

In [32]:
models["StrongCNN"] = StrongCNN()

models["StrongCNN"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "strong_cnn.pth"), map_location=device)
)

/tmp/ipykernel_41159/712774633.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "strong_cnn.pth"), map_location=device)


<All keys matched successfully>

In [33]:
models["ImprovedTinyCNN"] = ImprovedTinyCNN()

models["ImprovedTinyCNN"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "improved_tiny_cnn.pth"), map_location=device)
)

/tmp/ipykernel_41159/1863035296.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "improved_tiny_cnn.pth"), map_location=device)


<All keys matched successfully>

In [34]:
from torchvision.models import alexnet

alexnet_model = alexnet(weights=None)
alexnet_model.classifier[6] = nn.Linear(4096, 200)

alexnet_model.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "alexnet_finetuned.pth"), map_location=device)
)

models["AlexNet"] = alexnet_model

/tmp/ipykernel_41159/1757377088.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "alexnet_finetuned.pth"), map_location=device)


In [35]:
models["AlexNet3x3"] = AlexNet3x3(num_classes=200)

models["AlexNet3x3"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "alexnet3x3.pth"), map_location=device)
)

/tmp/ipykernel_41159/1397474959.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "alexnet3x3.pth"), map_location=device)


<All keys matched successfully>

In [36]:
print("\nFINAL MODEL COMPARISON")
print("-" * 50)

for name, model in models.items():
    acc = evaluate_model(model, val_loader, device)
    results[name] = acc
    print(f"{name:20s}: {acc:.2f}%")


FINAL MODEL COMPARISON
--------------------------------------------------
FastTinyCNN         : 18.95%
StrongCNN           : 48.87%
ImprovedTinyCNN     : 50.20%
AlexNet             : 53.86%
AlexNet3x3          : 42.95%


In [37]:
print("\nRANKING")
print("-" * 50)

for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:20s}: {acc:.2f}%")


RANKING
--------------------------------------------------
AlexNet             : 53.86%
ImprovedTinyCNN     : 50.20%
StrongCNN           : 48.87%
AlexNet3x3          : 42.95%
FastTinyCNN         : 18.95%
